# RAG walkthrough: step by step

This notebook traces through every stage of the pipeline, using the same code as the CLI (`rag/`), so what you see here is exactly what the system does.

| Stage | Section | Code |
|---|---|---|
| Load PDF pages | 1 | `rag/ingest.py` → `PyMuPDFLoader` |
| Label pages, split into chunks | 2 | `rag/ingest.py` → `page_kind`, `RecursiveCharacterTextSplitter` |
| Embeddings + cosine similarity | 3 | `rag/ingest.py` → `HuggingFaceEmbeddings` (bge-m3) |
| Retrieve (dense / BM25 / hybrid) | 4 | `rag/pipeline.py` → `Chroma`, `BM25Retriever`, `EnsembleRetriever` |
| Prompt | 5 | `rag/pipeline.py` → `RAG_PROMPT`, `format_context` |
| Generate + citations | 6 | `rag/pipeline.py` → `ChatOllama`, `answer` |
| Closed-book baseline | 7 | `rag/pipeline.py` → `answer_closed_book` |
| Evaluation | 8 | `rag/evaluate.py` |

**Before running:** `python -m rag.cli ingest` has been run once, and Ollama is running with `qwen3:8b` pulled. Run in VS Code / Jupyter with the `rag-lc` conda environment (`pip install ipykernel` if needed).

In [1]:
import os, sys
from pathlib import Path

# run from the repo root no matter where the notebook is opened
root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(root); sys.path.insert(0, str(root))

from rag.config import load_config, load_manifest, resolve
cfg = load_config()
QUESTION = "What is the MMVP benchmark?"   # change this to trace another question
cfg["retrieval"], cfg["generation"]["model"]

({'method': 'hybrid',
  'top_k': 5,
  'candidate_k': 20,
  'hybrid_weights': [0.5, 0.5],
  'bm25_tokenizer': 'words_lowercase'},
 'qwen3:8b')

## 1. Load: satu `Document` per halaman PDF
`PyMuPDFLoader` menghasilkan satu `Document` LangChain untuk setiap halaman. `metadata["page"]` dimulai dari **0**, jadi kode menambahkan 1 agar sama dengan nomor halaman yang terlihat di PDF viewer.

In [2]:
from langchain_community.document_loaders import PyMuPDFLoader

paper = next(p for p in load_manifest(cfg) if p["id"] == "P1")
pages = PyMuPDFLoader(str(resolve(cfg["paths"]["raw_dir"]) / "P1.pdf")).load()
print(f"{paper['id']}: {len(pages)} pages")
print("metadata of first page:", {k: pages[0].metadata[k] for k in ("page", "total_pages")})
print("\n--- page 1, first 500 characters ---\n", pages[0].page_content[:500])

C:\Users\blurtie\AppData\Local\Temp\ipykernel_35316\3050612768.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyMuPDFLoader


P1: 18 pages
metadata of first page: {'page': 0, 'total_pages': 18}

--- page 1, first 500 characters ---
 Eyes Wide Shut? Exploring the Visual Shortcomings of Multimodal LLMs
Shengbang Tong1
Zhuang Liu2
Yuexiang Zhai3
Yi Ma3
Yann LeCun1
Saining Xie1
1New York University
2FAIR, Meta
3UC Berkeley
The butterfly's feet are not visible in this 
image. The wings and part of the body of the 
butterfly are in view, but the feet are either 
hidden beneath the body or out of the frame of 
the picture. Butterflies have delicate, slender feet 
which are often not the most noticeable feature 
when they are perch


## 2. Label halaman dan pecah jadi chunk
Setiap halaman diberi label `main`, `references`, atau `appendix` berdasarkan nomor halaman yang dicek manual di `papers.yaml`. Halaman references dibuang. Halaman sisanya dipecah dengan `RecursiveCharacterTextSplitter` (ukuran dihitung dalam token bge-m3), sehingga **satu chunk tidak pernah melewati dua halaman**.

In [3]:
from rag.ingest import page_kind, make_splitter

print("P1 page labels:", {n: page_kind(n, paper) for n in range(1, len(pages) + 1)})
splitter = make_splitter(cfg)
p2_chunks = splitter.split_text(pages[1].page_content)
print(f"\npage 2 -> {len(p2_chunks)} chunks (chunk_size={cfg['chunking']['chunk_size']}, overlap={cfg['chunking']['chunk_overlap']} tokens)")
print("\n--- first chunk of page 2 ---\n", p2_chunks[0][:600])

P1 page labels: {1: 'main', 2: 'main', 3: 'main', 4: 'main', 5: 'main', 6: 'main', 7: 'main', 8: 'main', 9: 'references', 10: 'references', 11: 'references', 12: 'appendix', 13: 'appendix', 14: 'appendix', 15: 'appendix', 16: 'appendix', 17: 'appendix', 18: 'appendix'}


c:\Users\blurtie\anaconda3\envs\rag-lc\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



page 2 -> 5 chunks (chunk_size=400, overlap=80 tokens)

--- first chunk of page 2 ---
 1. Introduction
Multimodal Large Language Models (MLLMs) [8, 13, 31,
40] have been rapidly developing in recent times. MLLMs
integrate images into large language models (LLMs) and
leverage the powerful abilities of LLMs [41, 59, 69], show-
casing remarkable proficiency in tasks such as image under-
standing, visual question answering, and instruction follow-
ing. In particular, the recently released GPT-4V(ision) [40]
has pushed performance to an unprecedented level [41, 63].
Beneath the advancements of these models, we find there
exists a notable weakness: they still exhibit visual short-
com


## 3. Embedding dan cosine similarity
Teks → vektor, lalu bandingkan vektornya. Vektor bge-m3 dinormalisasi ke panjang 1, jadi dot product **sama dengan** cosine similarity.

In [4]:
import numpy as np
from rag.ingest import make_embeddings

emb = make_embeddings(cfg)
q_vec = np.array(emb.embed_query(QUESTION))
c_vecs = np.array(emb.embed_documents(p2_chunks[:3]))
print("vector size:", q_vec.shape[0], "| length of question vector:", round(float(np.linalg.norm(q_vec)), 3))
for i, score in enumerate(c_vecs @ q_vec):
    print(f"cosine(question, page-2 chunk {i}) = {score:.3f}")

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 28942.06it/s]


vector size: 1024 | length of question vector: 1.0
cosine(question, page-2 chunk 0) = 0.400
cosine(question, page-2 chunk 1) = 0.461
cosine(question, page-2 chunk 2) = 0.596


## 4. Retrieval: dense vs BM25 vs hybrid
- **dense**: similarity search di Chroma memakai vektor bge-m3 (berdasarkan makna).
- **bm25**: pencocokan keyword (`bm25_tokenizer` di `config.yaml`).
- **hybrid**: `EnsembleRetriever` = weighted Reciprocal Rank Fusion dari kedua daftar: `score = Σ weight / (60 + rank)`.

In [5]:
from rag.pipeline import RagPipeline

rag = RagPipeline(cfg, embeddings=emb)
for method in ("dense", "bm25", "hybrid"):
    print(f"\n=== {method} ===")
    for h in rag.search(QUESTION, method=method):
        c = h["chunk"]
        print(f"#{h['rank']} {c['chunk_id']:<12} {c['doc_id']} p.{c['pages'][0]:<3} {c['kind']:<9} {c['text'][:90]!r}")


=== dense ===
#1 P1-p02-2     P1 p.2   main      'We discover that these CLIP-blind pairs indeed lead to\nerrors in downstream MLLMs. With th'
#2 P1-p01-3     P1 p.1   main      'similar despite their clear visual differences. With these\npairs, we construct the Multimo'
#3 P1-p12-1     P1 p.12  appendix  'stage 2 uses DataMix 665k [1, 15, 21, 23, 24, 31, 34, 35, 38, 49, 51]\nproposed in Liu et a'
#4 P1-p15-0     P1 p.15  appendix  'method\nres\n#tokens\nMMVP\nLLVB\nLLVW\nMMB\nVQAT\nPOPE\nVQAV2\nMM-V\nLLaVA1.5\n3362\n576\n24.7\n84.7\n70.'
#5 P1-p08-1     P1 p.8   main      'aims to address through a systematic study.\nEvaluating Multimodal LLMs.\nMMVP assesses\nMLLM'

=== bm25 ===
#1 P1-p13-0     P1 p.13  appendix  'Questionnaire\nProgress:\n6/300\nCan you see the key "Z" in the image?\n Yes\n No\n This questio'
#2 P1-p04-2     P1 p.4   main      '(b) No\n(b)\n(a)\n\uf0fb\n(a)\n(b)\n✓\n(a)\n(b)\n✓\n(a)\n(a)\n\uf0fb\nIs the following statement correct: There a'
#3 P1-p05-0     P1 p.5  

**Apa yang terlihat dari output ini.** Dense dan hybrid menaruh pendahuluan MMVP (P1 hal. 2) di peringkat pertama. BM25 justru memberi peringkat tertinggi ke chunk gambar dan kuesioner (hal. 13, 4, 16): teks dari gambar yang terekstrak sebagai token lepas, cocok secara keyword tapi tidak menjelaskan apa itu MMVP. Ini kelemahan #4 di README (teks gambar di keyword search), dan alasan memakai hybrid, bukan BM25 saja.

## 5. Prompt yang benar-benar diterima LLM
Chunk hasil retrieval diberi nomor `[C1]..[C5]`. System message menyuruh model hanya memakai passage ini dan wajib mencantumkan sitasinya.

In [6]:
from rag.pipeline import RAG_PROMPT, format_context

docs = rag.retrieve(QUESTION)          # default method from config.yaml
messages = RAG_PROMPT.format_messages(context=format_context(docs), question=QUESTION)
for m in messages:
    print(f"----- {m.type.upper()} -----\n{m.content[:2500]}\n")

----- SYSTEM -----
You answer questions about a fixed collection of research papers.
Rules:
1. Use ONLY the information in the numbered context passages. Do not add facts from outside knowledge.
2. Cite the passage(s) supporting each claim right after it, e.g. [C1] or [C2][C4].
3. If the passages do not contain enough information to answer, say so plainly instead of guessing.
4. Answer in the same language as the question. Be concise.

----- HUMAN -----
Context passages:

[C1] P1: Eyes Wide Shut? Exploring the Visual Shortcomings of Multimodal LLMs | p.2 | main
We discover that these CLIP-blind pairs indeed lead to
errors in downstream MLLMs. With these pairs, We intro-
duce the MultiModal Visual Patterns (MMVP) benchmark.
This benchmark is specifically designed to inquire about
differences in CLIP-blind pairs and evaluate the visual abil-
ities of state-of-the-art MLLMs with straightforward ques-
tions. We evaluate a variety of open-source [8, 30, 31, 71]
and closed-source models [13,

## 6. Generate jawaban dan petakan sitasi ke halaman
`answer()` menjalankan `prompt | ChatOllama | StrOutputParser`, lalu memetakan setiap `[Cn]` di jawaban ke chunk yang diberikan sebagai passage ke-*n*. Nomor yang tidak pernah diberikan dilaporkan sebagai invalid citation. (Panggilan pertama bisa memakan ~90 detik karena Ollama memuat model.)

In [7]:
result = rag.answer(QUESTION, docs)
print(f"ANSWER ({result['seconds']} s):\n{result['content']}\n")
for s in result["sources"]:
    print(f"[{s['label']}] {s['doc_id']} p.{s['pages'][0]}  ({s['chunk_id']})  {s['title']}")
print("invalid citations:", result["invalid_citations"] or "none")

ANSWER (16.5 s):
The MMVP benchmark is a dataset designed to evaluate the visual abilities of state-of-the-art multimodal large language models (MLLMs) by focusing on CLIP-blind pairs, which are image-text pairs that are visually distinct but semantically similar. This benchmark is specifically crafted to identify and assess the challenges MLLMs face with straightforward visual questions, highlighting issues such as orientation, counting, and viewpoint [C1].

[C1] P1 p.2  (P1-p02-2)  Eyes Wide Shut? Exploring the Visual Shortcomings of Multimodal LLMs
invalid citations: none


**Mengecek jawaban ini terhadap passage yang disitasi (faithfulness).** Retrieval-nya bagus: [C1] adalah passage yang tepat. Tapi ada satu klaim yang **tidak ada** di [C1] dan juga salah: *"CLIP-blind pairs, which are image-text pairs that are visually distinct but semantically similar"*. [C1] tidak mendefinisikan CLIP-blind pairs; di paper, itu adalah pasangan **gambar** yang dianggap mirip oleh CLIP padahal tampilannya jelas berbeda (P1 hal. 1–3). Model mengisi kekosongan dengan tebakannya sendiri (yang salah).

Jadi ini masalah **generation**, bukan retrieval: evidence-nya sudah ter-retrieve, tapi model menambahkan klaim tanpa dukungan, alih-alih bilang bahwa passage tidak mendefinisikan istilah itu. Pertanyaan ini tidak termasuk evaluation set; contoh ini menunjukkan kenapa faithfulness dilabel terpisah dari correctness, dan kenapa pengecekan faithfulness otomatis masuk ke next steps.

## 7. Baseline closed-book
Model yang sama, **tanpa** passage hasil retrieval. Kalau jawaban ini juga benar, berarti model mungkin sudah tahu jawabannya dari pretraining, dan jawaban RAG yang benar tidak membuktikan bahwa retrieval bekerja.

In [8]:
closed = rag.answer_closed_book(QUESTION)
print(f"CLOSED-BOOK ({closed['seconds']} s):\n{closed['content']}")

CLOSED-BOOK (2.4 s):
The MMVP (Multi-Modal Video Prediction) benchmark is a dataset and evaluation framework used to assess the performance of models in predicting future frames in video sequences. It includes multiple modalities such as video, audio, and text, and is designed to evaluate both the visual and auditory aspects of video prediction. The benchmark provides a standardized way to compare different models across various tasks, including short-term and long-term video prediction.


**Hasil closed-book.** Tanpa konteks, model mengarang: MMVP dijawab sebagai *"Multi-Modal Video Prediction"* untuk prediksi frame video, padahal di koleksi ini MMVP = *MultiModal Visual Patterns*. Jadi jawaban RAG di bagian 6 (yang menyebut CLIP-blind pairs dan pattern seperti orientation, counting, viewpoint) memang berasal dari passage hasil retrieval, bukan dari pengetahuan model.

## 8. Evaluasi retrieval pada 12 pertanyaan yang bisa dijawab
Script yang sama dengan `python -m rag.evaluate retrieval`. Sebuah chunk dihitung *hit* jika berasal dari paper gold **dan** halaman gold. Untuk hasil generation: jalankan `python -m rag.evaluate generate`, label CSV secara manual, lalu `python -m rag.evaluate summarize <csv>`.

In [9]:
from rag.evaluate import eval_retrieval
eval_retrieval(cfg)

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 27199.15it/s]



Retrieval over 12 answerable questions (top-10 searched, metrics at k shown)
method    hit@1  hit@3  hit@5  MRR@10  allDocs@5  offDocAppx@5
dense      0.67   0.75   0.75    0.74       0.67          0.00
bm25       0.67   0.92   0.92    0.76       0.83          0.00
hybrid     0.67   0.75   1.00    0.76       0.83          0.00
offDocAppx@5 = average number of top-5 chunks that are appendix chunks of a NON-gold paper

hit@5 per question language (count / n):
  dense    en: 5/5   id: 4/7
  bm25     en: 4/5   id: 7/7
  hybrid   en: 5/5   id: 7/7

[saved] D:\Documents\Projects\Kalbe AI Engineer Intern\kalbe-rag-lc\outputs\eval\retrieval_20261019-194008.csv
[saved] D:\Documents\Projects\Kalbe AI Engineer Intern\kalbe-rag-lc\outputs\eval\retrieval_20261019-194008.json
